# Bonus: submit a real training run on Aurora

**This notebook is entirely optional.** `01_scaling_analysis.ipynb` does not depend on it in any way. If you have Aurora (ALCF) access, this notebook submits one real IsoFLOP training run to the `debug` queue, polls for it to finish, and folds the result into the scaling-law fit from the core notebook. If the job hasn't finished by the time we check, we fall back to a stored example point so the exercise doesn't block on a shared, busy queue.

**Requirements:** an Aurora account with access to the `ModCon` project allocation, and the `nanochat` submodule checked out at `../nanochat` (run `git submodule update --init` from the repo root if you haven't already).

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from scripts import aurora_job_helpers as helpers
from scaling_lib import core
import pandas as pd

if not helpers.on_pbs_system():
    print("No PBS (qsub/qstat) found on this machine — this notebook requires an Aurora login or "
          "compute node. Nothing else in this repo depends on this notebook; you're all set with "
          "01_scaling_analysis.ipynb.")
else:
    print("PBS detected — you can submit a job.")

## Submit one IsoFLOP point

This submits a single small training run (`depth=10`, `target_flops=5e16`) to the `debug` queue (1 node, ≤55 min walltime) billed to the `ModCon` allocation. It intentionally targets a budget smaller than any in `data/results.csv` so it doesn't just duplicate an existing point.

In [ ]:
DEPTH = 10
TARGET_FLOPS = "5e16"

job_id = None
if helpers.on_pbs_system():
    job_id = helpers.submit_job(depth=DEPTH, target_flops=TARGET_FLOPS)
    print(f"Submitted job {job_id} (depth={DEPTH}, target_flops={TARGET_FLOPS})")
else:
    print("Skipping submission — no PBS on this machine.")

## Poll for completion (bounded wait)

We poll a few times with a short interval rather than blocking indefinitely — the `debug` queue is shared, so the job may still be queued behind other users when we check. If it isn't done yet, we move on to the fallback below instead of stalling the room.

In [ ]:
job_finished = False
if job_id:
    job_finished = helpers.wait_for_job(job_id, poll_seconds=30, max_polls=6)
    print("Job finished" if job_finished else "Job still queued/running after the bounded wait — using fallback data below.")

## Get a result: live if finished, stored fallback otherwise

This is the key reliability property for a large-audience session: whether or not your specific job completed in time, you end up with a labeled data point and can continue the exercise.

In [ ]:
bonus_row = None
source_label = "stored example (not your live result)"

if job_finished:
    bonus_row = helpers.parse_result_row(TARGET_FLOPS, DEPTH)
    if bonus_row is not None:
        source_label = "live Aurora run"

if bonus_row is None:
    fallback_df = pd.read_csv(Path("..") / "data" / "bonus_fallback_point.csv")
    bonus_row = fallback_df.iloc[0].to_dict()

print(f"Using: {source_label}")
bonus_row

## Refit the scaling law with this extra point

Append the bonus point to the 24 runs from the core notebook and refit — same `scaling_lib.core` functions as before.

In [ ]:
df = core.load_raw()
bonus_df = pd.DataFrame([bonus_row])
bonus_df["flops_budget"] = bonus_df["flops_budget"].astype(float)
combined = pd.concat([df, bonus_df], ignore_index=True)
combined = core.compute_effective_params(combined)

winners = core.best_per_budget(combined)
slope, coeff = core.fit_power_law(winners["flops_budget"].values, winners["val_bpb"].values)
print(f"Fit including the bonus point ({source_label}): val_bpb = {coeff:.4f} * C^{slope:.4f}")
winners[["flops_budget", "depth", "effective_params", "val_bpb"]]

## Done

Whether your run finished live or you used the stored fallback, you've now seen the full loop: submit → (maybe) wait → fold a new data point into the same analysis from `01_scaling_analysis.ipynb`. This is exactly the workflow a real scaling-law study repeats at larger and larger budgets.